# Using the LangChain Transformer

LangChain is a software development framework designed to simplify the creation of applications using large language models (LLMs). Chains in LangChain go beyond just a single LLM call and are sequences of calls (can be a call to an LLM or a different utility), automating the execution of a series of calls and actions.
To make it easier to scale up the LangChain execution on a large dataset, we have integrated LangChain with the distributed machine learning library [SynapseML](https://www.microsoft.com/en-us/research/blog/synapseml-a-simple-multilingual-and-massively-parallel-machine-learning-library/). This integration makes it easy to use the [Apache Spark](https://spark.apache.org/) distributed computing framework to process millions of data with the LangChain Framework.

This tutorial shows how to apply LangChain at scale for paper summarization and organization. We start with a table of arxiv links and apply the LangChain Transformerto automatically extract the corresponding paper title, authors, summary, and some related works.

## Step 1: Prerequisites

The key prerequisites for this quickstart include a working Azure OpenAI resource, and an Apache Spark cluster with SynapseML installed. We suggest creating a Synapse workspace, but an Azure Databricks, HDInsight, or Spark on Kubernetes, or even a python environment with the `pyspark` package will work. 

1. An Azure OpenAI resource – request access [here](https://customervoice.microsoft.com/Pages/ResponsePage.aspx?id=v4j5cvGGr0GRqy180BHbR7en2Ais5pxKtso_Pz4b1_xUOFA5Qk1UWDRBMjg0WFhPMkIzTzhKQ1dWNyQlQCN0PWcu) before [creating a resource](https://docs.microsoft.com/en-us/azure/cognitive-services/openai/how-to/create-resource?pivots=web-portal#create-a-resource)
1. [Create a Synapse workspace](https://docs.microsoft.com/en-us/azure/synapse-analytics/get-started-create-workspace)
1. [Create a serverless Apache Spark pool](https://docs.microsoft.com/en-us/azure/synapse-analytics/get-started-analyze-spark#create-a-serverless-apache-spark-pool)

## Step 2: Import this guide as a notebook

The next step is to add this code into your Spark cluster. You can either create a notebook in your Spark platform and copy the code into this notebook to run the demo. Or download the notebook and import it into Synapse Analytics

1. Import the notebook into [Microsoft Fabric](https://learn.microsoft.com/en-us/fabric/data-engineering/how-to-use-notebook), [Synapse Workspace](https://docs.microsoft.com/en-us/azure/synapse-analytics/spark/apache-spark-development-using-notebooks#create-a-notebook) or if using Databricks into the [Databricks Workspace](https://docs.microsoft.com/en-us/azure/databricks/notebooks/notebooks-manage#create-a-notebook).
1. Install SynapseML on your cluster. Please see the installation instructions for Synapse at the bottom of [the SynapseML website](https://microsoft.github.io/SynapseML/). Note that this requires pasting an additional cell at the top of the notebook you just imported.
1. Connect your notebook to a cluster and follow along, editing and running the cells below.

In [ ]:
%pip install -U openai==2.47.0 langchain-openai==1.4.0 langchain-community==0.4.2 pdf2image pdfminer.six unstructured pytesseract nltk

In [ ]:
from langchain_community.document_loaders import OnlinePDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from synapse.ml.services.langchain import LangchainTransformer
from synapse.ml.core.platform import find_secret

##  Step 3: Fill in the service information and construct the LLM
Next, edit the cell to point to your Azure OpenAI v1 endpoint and deployment. You can replace `find_secret` with your key as follows:

`openai_api_key = "99sj2w82o...."`

In [ ]:
openai_api_key = find_secret(
    secret_name="openai-api-key-3", keyvault="mmlspark-build-keys"
)
openai_base_url = "https://synapseml-openai-3.openai.azure.com/openai/v1/"
deployment_name = "gpt-5-mini"

llm = ChatOpenAI(
    model=deployment_name,
    base_url=openai_base_url,
    api_key=openai_api_key,
    max_completion_tokens=1024,
    reasoning_effort="low",
)

## Step 4: Basic Usage of LangChain Transformer

### Create a chain
We will start by demonstrating the basic usage with a simple chain that creates definitions for input words

In [ ]:
copy_prompt = PromptTemplate(
    input_variables=["technology"],
    template="Define the following word: {technology}",
)

chain = (
    RunnableLambda(lambda technology: {"technology": technology})
    | copy_prompt
    | llm
    | StrOutputParser()
)
transformer = (
    LangchainTransformer()
    .setInputCol("technology")
    .setOutputCol("definition")
    .setChain(chain)
    .setSubscriptionKey(openai_api_key)
    .setUrl(openai_base_url)
)

### Create a dataset and apply the chain

In [ ]:
# construction of test dataframe
df = spark.createDataFrame(
    [(0, "docker"), (1, "spark"), (2, "python")], ["label", "technology"]
)
display(transformer.transform(df))

### Serialization note
This pipeline contains a `RunnableLambda`, which LangChain does not serialize. Use named serializable Runnables if persistence is required.

## Step 5: Using LangChain for Large scale literature review

### Create a Runnable pipeline for paper summarization

We will construct a Runnable pipeline that loads an arXiv PDF and extracts its title, authors, and a short summary.

The pipeline contains these steps:

1. **RunnableLambda**: Load the first two PDF pages.
2. **PromptTemplate**: Request the title, authors, and summary.
3. **ChatOpenAI**: Generate the structured paper description.
4. **StrOutputParser**: Return plain text to Spark.

In [ ]:
def paper_content_extraction(arxiv_link: str) -> dict:
    loader = OnlinePDFLoader(arxiv_link)
    pages = loader.load_and_split()
    content = "\n".join(page.page_content for page in pages[:2])
    return {"paper_content": content}

paper_summarizer_template = """Extract the paper title, authors, and a concise summary.
Here is the paper content:
{paper_content}
"""
paper_prompt = PromptTemplate.from_template(paper_summarizer_template)
paper_summary_chain = (
    RunnableLambda(paper_content_extraction)
    | paper_prompt
    | llm
    | StrOutputParser()
)

### Apply the LangChain transformer to perform this workload at scale

We can now use our chain at scale using the `LangchainTransformer`

In [ ]:
paper_df = spark.createDataFrame(
    [
        (0, "https://arxiv.org/pdf/2107.13586.pdf"),
        (1, "https://arxiv.org/pdf/2101.00190.pdf"),
        (2, "https://arxiv.org/pdf/2103.10385.pdf"),
        (3, "https://arxiv.org/pdf/2110.07602.pdf"),
    ],
    ["label", "arxiv_link"],
)

# construct langchain transformer using the paper summarizer chain define above
paper_info_extractor = (
    LangchainTransformer()
    .setInputCol("arxiv_link")
    .setOutputCol("paper_info")
    .setChain(paper_summary_chain)
    .setSubscriptionKey(openai_api_key)
    .setUrl(openai_base_url)
)


# Extract the paper title, authors, and a brief summary from each arXiv link.
display(paper_info_extractor.transform(paper_df))